In [1]:
import numpy as np

# ==========================================
# 1. GÉNÉRATION DES DONNÉES FINANCIÈRES FICTIVES
# ==========================================
np.random.seed(42) 
n_samples = 100 
n_features = 3  # On passe à 3 features pour tester la sélection de variables

X = np.random.randn(n_samples, n_features)

# On définit les vrais coefficients : la feature 3 ne sert à RIEN (coefficient = 0)
# beta_0 (intercept) = 2.0, beta_1 = 3.5, beta_2 = -1.5, beta_3 = 0.0
true_intercept = 2.0
true_beta = np.array([3.5, -1.5, 0.0])

# Génération de Y avec un bruit gaussien
Y = true_intercept + np.dot(X, true_beta) + np.random.randn(n_samples) * 0.5

# ==========================================
# 2. FONCTION DE SEUIL DOUX (SOFT-THRESHOLDING)
# ==========================================
def soft_thresholding(rho, lam):
    """ C'est le cœur du LASSO qui force les coefficients à zéro """
    if rho < -lam:
        return rho + lam
    elif rho > lam:
        return rho - lam
    else:
        return 0.0

# ==========================================
# 3. ALGORITHME LASSO (DESCENTE DE COORDONNÉES)
# ==========================================
def lasso_coordinate_descent(X, y, lam, num_iters=500, tol=1e-6):
    n_samples, n_features = X.shape
    
    # 1. On initialise l'intercept et les coefficients à zéro
    intercept = 0.0
    beta = np.zeros(n_features)
    
    for it in range(num_iters):
        beta_old = beta.copy()
        
        # Mettre à jour l'intercept (moyenne des résidus)
        intercept = np.mean(y - np.dot(X, beta))
        
        # Mettre à jour chaque coefficient un par un
        for j in range(n_features):
            # Prédiction sans la feature j (en incluant l'intercept actuel)
            y_pred_without_j = intercept + np.dot(X, beta) - beta[j] * X[:, j]
            
            # Résidu (l'erreur que le modèle doit corriger)
            r = y - y_pred_without_j
            
            # rho correspond à la corrélation entre la feature j et le résidu
            rho = np.dot(X[:, j], r)
            
            # Application du Soft-Thresholding
            denominator = np.sum(X[:, j]**2)
            beta[j] = soft_thresholding(rho, lam * n_samples) / denominator
            
        # Condition d'arrêt si l'algorithme a convergé
        if np.linalg.norm(beta - beta_old) < tol:
            break
            
    return intercept, beta

# ==========================================
# 4. CALCUL ET AFFICHAGE DES RÉSULTATS
# ==========================================
lam = 0.1  # Paramètre de régularisation (lambda)

intercept_calcule, beta_calcule = lasso_coordinate_descent(X, Y, lam)

print("="*50)
print("--- COMPARAISON DES RÉSULTATS ---")
print("Vrai Intercept :", true_intercept, " | Calculé :", round(intercept_calcule, 3))
print("Vrais Betas    :", true_beta)
print("Betas LASSO    :", np.round(beta_calcule, 3))
print("="*50)

--- COMPARAISON DES RÉSULTATS ---
Vrai Intercept : 2.0  | Calculé : 2.084
Vrais Betas    : [ 3.5 -1.5  0. ]
Betas LASSO    : [ 3.328 -1.418  0.   ]
